<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-2 · Part 2: Inner Products and Projections

**An inner product measures alignment between cooling displacements and determines projection onto a line.**

Part 1 represented cooling by \\(u=(u_{\mathrm{early}},u_{\mathrm{late}})\\). The same coordinate order applies to displacement vectors. This unit adds inner products and projections; the comparison concerns vector geometry rather than simulated performance.

### 1 · Inner products and angles

For two real vectors \\(v,w\in\mathbb R^p\\), where \\(p\\) is the number of coordinates (two here), their Euclidean inner product (dot product) is a scalar:

> $\displaystyle v^{\mathsf T}w=\sum_{i=1}^{p}v_iw_i.$

The Euclidean length is \\(\lVert v\rVert_2=\sqrt{v^{\mathsf T}v}\\). For nonzero vectors, the angle \\(\theta\\) between them satisfies

> $\displaystyle v^{\mathsf T}w=\lVert v\rVert_2\lVert w\rVert_2\cos\theta.$

| Dot product | Angle between nonzero vectors | Interpretation |
|:---|:---|:---|
| Positive | Less than 90° | An aligned component |
| Zero | 90° | Perpendicular (orthogonal) |
| Negative | More than 90° | An opposing component |

A zero vector has zero dot product with every vector, but no defined angle. A zero dot product also does not by itself imply statistical independence.

For cooling changes \\(v=(1,0)\\) and \\(w=(0.5,-0.5)\\), \\(v^{\mathsf T}w=0.5\\): both include an increase in early cooling. This says nothing yet about whether the classroom score improves.

In NumPy, <code>v @ w</code> computes the dot product; <code>v * w</code> returns coordinatewise products.

### 2 · Projection onto a line

The projection onto the line through a nonzero \\(v\\) extracts the part of \\(w\\) along that line:

> $\displaystyle \operatorname{proj}_{v}(w)=\frac{v^{\mathsf T}w}{v^{\mathsf T}v}\,v.$

The residual \\(w-\operatorname{proj}_v(w)\\) is perpendicular to \\(v\\). Projecting onto a line does not by itself find a feasible cooling decision.

### 3 · Angle dependence

The comparison fixes both vector lengths at one and varies only their angle. The reference \\(v=(1,0)\\) represents the early-cooling direction; \\(w=(\cos\theta,\sin\theta)\\) is a displacement direction.

In [ ]:
import sys
import warnings

import matplotlib
import numpy as np
from matplotlib.lines import Line2D


def _pyplot(*, interactive=False):
    """Use ipympl outside the Playground, with a static fallback."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (ImportError, RuntimeError, ValueError):
            try:
                matplotlib.use("module://ipympl.backend_nbagg", force=True)
            except (ImportError, RuntimeError, ValueError):
                warnings.warn("Interactive backend unavailable; showing a static preview.")
    import matplotlib.pyplot as plt
    return plt


BLUE, TEAL, ORANGE = "#2563EB", "#0F8B7C", "#E88726"
PURPLE, GRAY = "#7C3AED", "#9CA3AF"


def style_axis(axis):
    axis.grid(alpha=0.25)
    axis.spines[["top", "right"]].set_visible(False)

In [ ]:
def draw_alignment(axis, angle_degrees):
    axis.clear()
    angle = np.radians(angle_degrees)
    reference = np.array([1.0, 0.0])
    direction = np.array([np.cos(angle), np.sin(angle)])
    projection = (reference @ direction) / (reference @ reference) * reference
    circle = np.linspace(0, 2 * np.pi, 361)
    axis.plot(np.cos(circle), np.sin(circle), color=GRAY, alpha=0.45)
    for vector, color in [(reference, BLUE), (direction, ORANGE)]:
        axis.annotate("", xy=vector, xytext=(0, 0),
                      arrowprops=dict(arrowstyle="->", color=color, lw=3))
    axis.plot([direction[0], projection[0]], [direction[1], projection[1]],
              "--", color=GRAY)
    axis.plot([0, projection[0]], [0, projection[1]], color=BLUE, lw=6, alpha=0.3)
    axis.scatter(*projection, color=BLUE, marker="s", s=45)
    axis.set(xlim=(-1.4, 1.4), ylim=(-1.35, 1.35), aspect="equal",
             xlabel="Early-cooling direction component", ylabel="Late-cooling direction component",
             title=f"Angle = {angle_degrees:.0f}°; dot product = {reference @ direction:+.3f}")
    axis.legend(handles=[Line2D([], [], color=BLUE, lw=2, label="Reference v"),
                         Line2D([], [], color=ORANGE, lw=2, label="Direction w"),
                         Line2D([], [], color=BLUE, marker="s", linestyle="None",
                                label="Projection onto v")], loc="lower left", fontsize=8)
    style_axis(axis)


def show_inner_product():
    plt = _pyplot()
    figure, axes = plt.subplots(1, 3, figsize=(12.0, 4.3))
    for axis, angle in zip(axes, [45, 90, 135]):
        draw_alignment(axis, angle)
    figure.tight_layout()
    return figure


def show_alignment_explorer():
    plt = _pyplot(interactive=True)
    from matplotlib.widgets import Slider
    figure, axis = plt.subplots(figsize=(7.0, 6.1))
    figure.subplots_adjust(left=0.15, right=0.95, top=0.9, bottom=0.24)
    angle_slider = Slider(figure.add_axes([0.22, 0.10, 0.62, 0.04]),
                          "Angle (°)", 0, 180, valinit=45, valstep=1, color=ORANGE)
    def update(_):
        draw_alignment(axis, angle_slider.val)
        figure.canvas.draw_idle()
    angle_slider.on_changed(update)
    figure._sliders, figure._update = [angle_slider], update
    update(None)
    return figure

At 45°, 90°, and 135°, the dot product is positive, zero, and negative, respectively. Both lengths remain fixed, isolating the effect of alignment.


<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/02_inner_product_directional_change.svg" alt="Three equal-length vector pairs at acute, right, and obtuse angles with their projections" width="900" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

In [ ]:
reference = np.array([1.0, 0.0])
change = np.array([0.5, -0.5])
print("Coordinatewise products:", reference * change)
print("Dot product:", reference @ change)
alignment_figure = show_inner_product()
_pyplot().show()
_pyplot().close(alignment_figure)

The angle-dependent dot product is 0.5 at 60°, zero at 90°, and −0.5 at 120°. The angle parameter changes the direction geometry; no cooling decision is evaluated.


In [ ]:
alignment_explorer = show_alignment_explorer()
_pyplot().show()